# Build LCAT database from data

## Overview

* We can build the LCAT from scratch (i.e. using the raw data files) using the processing scripts in this notebook.
* Please start by reading the documentation in the `/docs` folder. This notebook will contain only light LCAT theory.
* After that, set up the data module virtual environment, and run these notebook cells.

## Requirements

* CHESS-SCAPE data files covering:
    * RCPs 6.0 and 8.5
    * Seasonal and annual cases
    * Bias and non-bias corrected cases
    * Variables `pr`, `sfcWind`, `rsds`, `tas`, `tasmin`, `tasmax`
* Boundary shapefiles for:
    * UK Counties - `uk_counties`
    * Local Authority Districts - `la_districts`
    * LSOAs - `lsoa`
    * MSOAs - `msoa`
    * Parishes - `parishes`
    * Scotalnd Data Zones - `sc_dz`
    * Northern Ireland Data Zones - `ni_dz`
    * Isle of Man - `iom`
* A `config.yml` in the project root containing paths as described in the docs.
* Postgres should be installed on your system.
* Poetry should be installed on your system, and the virtual environment set up, at minimum with `poetry install` from the data module root, as described in the docs. 

Please note that this process has been tested on macOS.

## Initialise

* Load config and set up database/role.

In [ ]:
import os
import yaml

# The cwd should be the data folder root
os.chdir("..")

In [ ]:
config_filepath = "./config.yml"

with open(config_filepath) as f:
    conf = yaml.load(f, Loader=yaml.FullLoader)

# i.e...
db_config = {
    'superuser': conf["superuser"],
    'superuser_pass': conf["superuser_pass"],
    'host': conf["host"],
    'dbname': conf["dbname"],
    'user': conf["user"],
    'user_pass': conf["user_pass"],
}

## Create new database and role

In [ ]:
from src.db_manager import DBManager

In [ ]:
db_manager = DBManager(**db_config)

In [ ]:
# Setup db: create role, create database, add postgis extension
db_manager.setup_database()

# We also have some tests we can run
db_manager.test_new_user_capabilities()

## Load boundaries

In [ ]:
from src.boundary_loader import BoundaryLoader

In [ ]:
boundary_loader = BoundaryLoader(conf)

In [ ]:
boundary_loader.connect_to_db()

In [ ]:
boundary_loader.load_all_boundaries()

# Load CHESS-SCAPE grid

* First we need to load the grid cells (present in all of the NetCDF files).
* This class also identifies the bias and non bias corrected cells present.
* Finally, this class also identifies and tags grid cells as coastline, land, or 10km, 20km, 30km, 40km and 50km land from the coast.

In [ ]:
from src.grid_loader import GridLoader

In [ ]:
grid_loader = GridLoader(conf)

In [ ]:
grid_loader.connect_to_db()

In [ ]:
grid_loader.open_netcdf_files()

In [ ]:
grid_loader.process_grid()

## Load CHESS-SCAPE data

* Now we can load the climate variable data from the NetCDF files.

In [ ]:
from src.chessscape_loader import ChessScapeLoader

In [ ]:
# We need to provide the aggregated, labelled grid data to the ChessScapeLoader
# This will enable the loader to correctly select the bias and non bias corrected data
labelled_grid = grid_loader.masks["aggregated_labelled"]

In [ ]:
chess_loader = ChessScapeLoader(conf, labelled_grid)

In [ ]:
chess_loader.connect_to_db()

In [ ]:
chess_loader.process_all_rcps()

## Load CHESS-SCAPE averages

* Load UK averages for climate variables into the database.
* Note that this class contains lots of repeated code with the ChessScapeLoader class above: combining these classes will reduce database build times significantly.

In [ ]:
from src.chessscape_averages_loader import ChessScapeAveragesLoader

In [ ]:
uk_average_loader = ChessScapeAveragesLoader(conf)

In [ ]:
uk_average_loader.connect_to_db()

In [ ]:
uk_average_loader.process_all_data()

## Detect and store region/grid cell overlaps

* With the boundary regions, grid cells, and climate data loaded, we now need to find the overlapping grid cells for each region.

In [ ]:
from src.overlap_calculator import OverlapCalculator

In [ ]:
overlap_calculator = OverlapCalculator(conf)

In [ ]:
overlap_calculator.connect_to_db()

In [ ]:
overlap_calculator.process_all_boundary_overlaps(process_no_overlaps=True)

## Identify coastal regions

* In the LCAT tool, we want to show coastal-related impact pathways for coastal regions only. This means we need to identify which regions are coastal or not.
* This class uses the overlapping CHESS-SCAPE grid cells identified previously to tag regions as coastal or not.

In [ ]:
from src.coastal_identifier import CoastalIdentifier

In [ ]:
coastal_region_identifier = CoastalIdentifier(conf)

In [ ]:
coastal_region_identifier.connect_to_db()

In [ ]:
coastal_region_identifier.process_all_boundaries()

## Average and cache climate data

* In the LCAT tool, we want to serve climate predictions for selected regions with low latency. For regions with many overlapping grid cells, we average the climate data over these cells, and cache these averages.

In [ ]:
from src.cache_climate import CacheClimate

In [ ]:
cacher = CacheClimate(conf)

In [ ]:
cacher.connect_to_db()

In [ ]:
cacher.process_all_boundaries()

## Store boundary details

* We have some metadata to store.

In [ ]:
from src.boundary_details import DetailsGenerator

In [ ]:
generator = DetailsGenerator(conf)

In [ ]:
generator.connect_to_db()

In [ ]:
boundary_details = {
    "uk_counties": {
        "print_name": "UK Counties and Unitary Authorities",
        "shapefile_name_col": "CTYUA23NM",
        "source_srid": 27700,
        "db_srid": 27700,
        "method": "cache",
    },
    "la_districts": {
        "print_name": "LA Districts",
        "shapefile_name_col": "LAD23NM",
        "source_srid": 27700,
        "db_srid": 27700,
        "method": "cache",
    },
    "lsoa": {
        "print_name": "LSOA",
        "shapefile_name_col": "LSOA21NM",
        "source_srid": 27700,
        "db_srid": 27700,
        "method": "cell",
    },
    "msoa": {
        "print_name": "MSOA",
        "shapefile_name_col": "MSOA21NM",
        "source_srid": 27700,
        "db_srid": 27700,
        "method": "cell",
    },
    "parishes": {
        "print_name": "Parishes",
        "shapefile_name_col": "PAR23NM",
        "source_srid": 27700,
        "db_srid": 27700,
        "method": "cell",
    },
    "sc_dz": {
        "print_name": "Scotland Data Zones",
        "shapefile_name_col": "name",
        "source_srid": 27700,
        "db_srid": 27700,
        "method": "cell",
    },
    "ni_dz": {
        "print_name": "Northern Ireland Data Zones",
        "shapefile_name_col": "DZ2021_nm",
        "source_srid": 29902,
        "db_srid": 27700,
        "method": "cell",
    },
    "iom": {
        "print_name": "Isle of Man",
        "shapefile_name_col": "NAME_ENGLI",
        "source_srid": 4326,
        "db_srid": 27700,
        "method": "cache",
    },
}

In [ ]:
generator.process_data(boundary_details)

## Create references table

* We also back up the reference file (that we ship as a .json file in the front end) in a database table.
* This is not currently used, and really is just a mirror of the version in the Google Sheet.

In [ ]:
from src.reference_loader import ReferenceLoader

In [ ]:
reference_loader = ReferenceLoader(conf)

In [ ]:
reference_loader.load_all_references()

# Conclusion

We should now have a complete LCAT database for use locally. Ensure `server/.env` is populated with the same credentials as in `data/config.yml`, and run the application.